# Prepare IMC_NB_FineCT dataset

Creates a new HDF5 dataset with a less-stringent cell-type merging than the original `IMC_NeuroblastomaMetaCluster`.

**Key changes vs MetaCluster (7 classes → 11 classes):**
- T_Cell split into **CD4_T** / **CD8_T** / T_Cell (unknown subset)
- **NK_DC** separated from Myeloid (GZMB+ S100B-)
- **Neutrophil** separated from Myeloid
- **Proliferating** separated from Tumor (Ki67hi, LIN- Ki67+)
- 'Other' is present but set as `ignore_annotation` in the dataset config

Source: `annotations.csv` from `IMC_Neuroblastoma/` (original fine-grained labels from MetaCluster pipeline)

Output: `h5_files/IMC_NB_FineCT/IMC_NB_FineCT.h5` + `train.txt`, `val.txt`, `test.txt`, `used_markers.txt`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import h5py
import tifffile
from natsort import natsorted
from collections import Counter
from tqdm import tqdm

dataset_base  = Path('/home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data')
source_name   = 'IMC_Neuroblastoma'          # folder created by IMC_notebook.ipynb
new_name      = 'IMC_NB_FineCT'

source_path        = dataset_base / source_name
output_h5_folder   = dataset_base / f'h5_files/{new_name}'
output_h5_folder.mkdir(parents=True, exist_ok=True)
output_h5_path     = output_h5_folder / f'{new_name}.h5'

print(f'Source:  {source_path}')
print(f'Output:  {output_h5_path}')

Source:  /home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/IMC_Neuroblastoma
Output:  /home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/IMC_NB_FineCT/IMC_NB_FineCT.h5


## 1 · Load source annotations and show original distribution

In [2]:
annotations  = pd.read_csv(source_path / 'annotations.csv')
marker_names = pd.read_csv(source_path / 'marker_names.csv', index_col=0)['Label'].to_list()

print(f'Total cells: {len(annotations):,}')
print(f'Markers:     {len(marker_names)}')
print()
print('Original fine-grained distribution:')
vc = annotations['annotation'].value_counts()
for ct, n in vc.items():
    print(f'  {ct:>35}  {n:>7,}  ({100*n/len(annotations):5.2f}%)')

Total cells: 300,250
Markers:     44

Original fine-grained distribution:
                          early NB TC   49,538  (16.50%)
                  fibroblast/endothel   36,418  (12.13%)
                             CD44+ TC   33,873  (11.28%)
                           CXCR4hi TC   31,911  (10.63%)
                            Ki67hi TC   31,051  (10.34%)
                       bridge-like TC   24,200  ( 8.06%)
                   CD24- marker-lo TC   19,081  ( 6.36%)
                   CD24+ marker-lo TC   18,365  ( 6.12%)
                  dense T cell region   10,551  ( 3.51%)
                               B cell    6,715  ( 2.24%)
                             GD2lo TC    5,322  ( 1.77%)
                          CD3+ T cell    5,243  ( 1.75%)
                                other    4,094  ( 1.36%)
                           neutrophil    3,917  ( 1.30%)
                       monoblast-like    3,234  ( 1.08%)
                    CD14- PDL1+ MO/DC    3,018  ( 1.01%)
              

## 2 · Define new merging map

In [3]:
fine_ct_map = {
    # ── Tumor ──────────────────────────────────────────────────────────────
    # Classical NB tumour cells (CHGA, GATA3, GD2, S100B, SOX10 positive)
    'early NB TC':         'Tumor',
    'CHGAhi TC':           'Tumor',
    'GATA3hi TC':          'Tumor',
    'GD2lo TC':            'Tumor',
    'CXCR4hi TC':          'Tumor',
    'bridge-like TC':      'Tumor',
    'CD24- marker-lo TC':  'Tumor',
    'CD24+ marker-lo TC':  'Tumor',
    'CD44+ TC':            'Tumor',
    # Actively proliferating cells (Ki-67+) — kept separate
    'Ki67hi TC':           'Proliferating',
    'LIN- Ki67+':          'Proliferating',

    # ── Stromal ────────────────────────────────────────────────────────────
    'fibroblast/endothel': 'Stromal',
    'schwann cell':        'Stromal',

    # ── T cells — split CD4 / CD8 / unknown ───────────────────────────────
    'CD4+ naive T cell':        'CD4_T',
    'CD4+ PD1+ T cell':         'CD4_T',
    'CD8+ naive T cell':        'CD8_T',
    'CD8+ PDL1lo T cell':       'CD8_T',
    'CD8+ GZMB+ PD1lo T cell':  'CD8_T',
    'CD8+ S100B+ T cell':       'CD8_T',
    'CD3+ T cell':               'T_Cell',    # CD4/CD8 not assigned
    'CD3+ GZMB+ T cell':         'T_Cell',
    'dense T cell region':       'T_Cell',

    # ── NK / DC ────────────────────────────────────────────────────────────
    # GZMB+, S100B- → likely NK; separated from monocyte/macrophage myeloid
    'GZMB+ S100B- DC/NK':  'NK_DC',

    # ── Myeloid ────────────────────────────────────────────────────────────
    'neutrophil':         'Neutrophil',   # CD15+ — distinct from monocytes
    'monoblast-like':     'Myeloid',
    'CD14- PDL1+ MO/DC':  'Myeloid',
    'CD14+ PDL1+ MO':     'Myeloid',
    'CD14+ PDL1- MO':     'Myeloid',
    'pDC':                'Myeloid',

    # ── B cells ────────────────────────────────────────────────────────────
    'B cell':  'B_Cell',

    # ── Progenitors ────────────────────────────────────────────────────────
    'HPC':                'Progenitor',
    'neural progenitor':  'Progenitor',
    'Ki67+ CXCR4+ cell':  'Progenitor',

    # ── Other (will be ignored during training) ────────────────────────────
    'other':  'Other',
}

print(f'Mapping covers {len(fine_ct_map)} original cell types → {len(set(fine_ct_map.values()))} merged classes:')
from collections import defaultdict
reverse = defaultdict(list)
for k, v in fine_ct_map.items():
    reverse[v].append(k)
for cls in sorted(reverse):
    print(f'  {cls:<15}  ←  {", ".join(reverse[cls])}')

Mapping covers 34 original cell types → 12 merged classes:
  B_Cell           ←  B cell
  CD4_T            ←  CD4+ naive T cell, CD4+ PD1+ T cell
  CD8_T            ←  CD8+ naive T cell, CD8+ PDL1lo T cell, CD8+ GZMB+ PD1lo T cell, CD8+ S100B+ T cell
  Myeloid          ←  monoblast-like, CD14- PDL1+ MO/DC, CD14+ PDL1+ MO, CD14+ PDL1- MO, pDC
  NK_DC            ←  GZMB+ S100B- DC/NK
  Neutrophil       ←  neutrophil
  Other            ←  other
  Progenitor       ←  HPC, neural progenitor, Ki67+ CXCR4+ cell
  Proliferating    ←  Ki67hi TC, LIN- Ki67+
  Stromal          ←  fibroblast/endothel, schwann cell
  T_Cell           ←  CD3+ T cell, CD3+ GZMB+ T cell, dense T cell region, CD44+ TC
  Tumor            ←  early NB TC, CHGAhi TC, GATA3hi TC, GD2lo TC, CXCR4hi TC, bridge-like TC, CD24- marker-lo TC, CD24+ marker-lo TC


## 3 · Apply merging and check for unmapped types

In [4]:
annotations['annotation_new'] = annotations['annotation'].map(fine_ct_map)

unmapped = annotations.loc[annotations['annotation_new'].isna(), 'annotation'].unique()
if len(unmapped) > 0:
    print(f'WARNING — unmapped types (will be dropped): {unmapped}')
else:
    print('All cell types mapped successfully.')

ann_filtered = annotations.dropna(subset=['annotation_new']).copy()
print(f'\nCells before: {len(annotations):,}  →  after: {len(ann_filtered):,}  (dropped {len(annotations)-len(ann_filtered):,})')

print('\nNew distribution (including Other):')
vc_new = ann_filtered['annotation_new'].value_counts()
for ct, n in vc_new.items():
    print(f'  {ct:<15}  {n:>7,}  ({100*n/len(ann_filtered):5.2f}%)')

All cell types mapped successfully.

Cells before: 300,250  →  after: 300,250  (dropped 0)

New distribution (including Other):
  Tumor            150,531  (50.14%)
  T_Cell            49,752  (16.57%)
  Stromal           38,454  (12.81%)
  Proliferating     31,840  (10.60%)
  Myeloid            7,736  ( 2.58%)
  B_Cell             6,715  ( 2.24%)
  Other              4,094  ( 1.36%)
  Neutrophil         3,917  ( 1.30%)
  CD8_T              3,437  ( 1.14%)
  Progenitor         1,833  ( 0.61%)
  CD4_T              1,830  ( 0.61%)
  NK_DC                111  ( 0.04%)


## 4 · Create HDF5

In [5]:
sample_ids = natsorted(ann_filtered['SampleID'].unique().tolist())
print(f'Samples: {len(sample_ids)}')

annotation_strings = ann_filtered['annotation_new'].astype(str).tolist()

with h5py.File(output_h5_path, 'w') as h5f:
    dt = h5py.special_dtype(vlen=str)

    # Marker names
    marker_ds = h5f.create_dataset('marker_names', (len(marker_names),), dtype=dt)
    marker_ds[:] = marker_names

    # Coordinates and annotations (flat arrays, same order as ann_filtered)
    coords = h5f.create_group('coords')
    coords.create_dataset('DIM1',      data=ann_filtered['DIM1'].values)
    coords.create_dataset('DIM2',      data=ann_filtered['DIM2'].values)
    coords.create_dataset('sample_id', data=ann_filtered['SampleID'].astype(str).tolist(), dtype=dt)

    h5f.create_dataset('annotation', data=annotation_strings, dtype=dt)
    h5f.create_dataset('sample_ids', data=sample_ids, dtype=dt)

    # Image and mask data
    data_grp = h5f.create_group('data')
    for sid in tqdm(sample_ids, desc='Writing samples'):
        img   = tifffile.imread(source_path / f'images/{sid}_image.tif')
        masks = tifffile.imread(source_path / f'masks/{sid}_masks.tif').astype(np.uint32)
        sg = data_grp.create_group(sid)
        sg.create_dataset('image',  data=img,   compression='gzip', compression_opts=3, chunks=(64, 64, 1))
        sg.create_dataset('masks',  data=masks, compression='gzip', compression_opts=3, chunks=(64, 64))

print(f'\nSaved → {output_h5_path}')

Samples: 87


Writing samples: 100%|██████████| 87/87 [02:44<00:00,  1.90s/it]



Saved → /home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/IMC_NB_FineCT/IMC_NB_FineCT.h5


## 5 · Verify H5 structure

In [6]:
with h5py.File(output_h5_path, 'r') as f:
    print('Root keys:    ', list(f.keys()))
    print('Samples:      ', len(f['sample_ids']))
    print('Annotations:  ', len(f['annotation']))
    print('Unique classes:', np.unique(f['annotation'][()].astype(str)))
    sid0 = f['sample_ids'][0].decode()
    print(f'Example [{sid0}] image shape:', f['data'][sid0]['image'].shape)
    print(f'Example [{sid0}] masks shape:', f['data'][sid0]['masks'].shape)

Root keys:     ['annotation', 'coords', 'data', 'marker_names', 'sample_ids']
Samples:       87
Annotations:   300250
Unique classes: ['B_Cell' 'CD4_T' 'CD8_T' 'Myeloid' 'NK_DC' 'Neutrophil' 'Other'
 'Progenitor' 'Proliferating' 'Stromal' 'T_Cell' 'Tumor']
Example [20220925-20220804-17-001-TU-002] image shape: (696, 696, 44)
Example [20220925-20220804-17-001-TU-002] masks shape: (696, 696)


## 6 · Generate train / val / test splits (70 / 10 / 20, stratified per class)

In [7]:
annotations_arr = np.array(annotation_strings)
counts          = Counter(annotations_arr)
total           = len(annotations_arr)

splits = dict(train=0.7, val=0.1, test=0.2)
assert sum(splits.values()) == 1.0

train_idxs, val_idxs, test_idxs = [], [], []
rng = np.random.default_rng(42)

for celltype, n in counts.items():
    idxs = np.where(annotations_arr == celltype)[0]
    rng.shuffle(idxs)
    t = int(n * splits['train'])
    v = int(n * splits['val'])
    train_idxs.extend(sorted(idxs[:t]))
    val_idxs.extend(sorted(idxs[t:t+v]))
    test_idxs.extend(sorted(idxs[t+v:]))

assert len(train_idxs) + len(val_idxs) + len(test_idxs) == total

np.savetxt(output_h5_folder / 'train.txt', sorted(train_idxs), fmt='%d')
np.savetxt(output_h5_folder / 'val.txt',   sorted(val_idxs),   fmt='%d')
np.savetxt(output_h5_folder / 'test.txt',  sorted(test_idxs),  fmt='%d')

print(f'Train: {len(train_idxs):,}  |  Val: {len(val_idxs):,}  |  Test: {len(test_idxs):,}')
print('Saved train.txt, val.txt, test.txt')

Train: 210,169  |  Val: 30,021  |  Test: 60,060
Saved train.txt, val.txt, test.txt


## 7 · Create used_markers.txt

Same exclusion list as `IMC_NeuroblastomaMetaCluster` — removes low-quality / non-biological channels.

In [8]:
exclude = {
    'IF1', 'IF2', 'IF3',       # internal fiducials
    'MPO',                      # low SNR in IMC
    'H3K9Ac', 'H4K12Ac',       # histone marks — noisy
    'CXCR2',                    # low expression
    'IDO',                      # low expression
    'clPARP',                   # apoptosis — not cell-type specific
    'PNMT',                     # low SNR
    'DNA1',                     # structural, not protein
    'Fibronectin',              # ECM — not cell-type specific
    'FOXP3',                    # excluded: low SNR in IMC (nuclear TF)
}

with h5py.File(output_h5_path, 'r') as f:
    all_markers = set(f['marker_names'][()].astype(str))

kept = sorted(all_markers - exclude)
print(f'Total markers: {len(all_markers)}  →  kept: {len(kept)}  (excluded: {len(exclude)})')
print('Kept:', kept)

np.savetxt(output_h5_folder / 'used_markers.txt', kept, fmt='%s')
print('\nSaved used_markers.txt')

Total markers: 44  →  kept: 31  (excluded: 13)
Kept: ['CD10', 'CD11b', 'CD11c', 'CD14', 'CD15', 'CD20', 'CD24', 'CD274', 'CD279', 'CD3', 'CD34', 'CD4', 'CD44', 'CD45', 'CD56', 'CD8a', 'CHGA', 'CXCR4', 'DNA2', 'ELAVL4', 'GATA3', 'GD2', 'GZMB', 'HLA-ABC', 'HLA-DR', 'Ki-67', 'LUM', 'PRPH', 'S100B', 'SOX10', 'Vimentin']

Saved used_markers.txt


## 8 · Summary — dataset config values

In [9]:
n_markers = len(kept)
n_classes = len(set(fine_ct_map.values()) - {'Other'})

print('=' * 55)
print(f'  Dataset:    {new_name}')
print(f'  H5 path:    {output_h5_path}')
print(f'  n_markers:  {n_markers}   (same as IMC_NeuroblastomaMetaCluster)')
print(f'  n_classes:  {n_classes}   (excl. Other)')
print(f'  patch_size: 24   (same)')
print(f'  cutter_size:12   (same)')
print(f'  ignore_annotation: ["Other"]')
print('=' * 55)
print()
print('Cell type counts (excl. Other):')
for ct, n in vc_new.items():
    if ct == 'Other':
        continue
    print(f'  {ct:<15}  {n:>7,}')

  Dataset:    IMC_NB_FineCT
  H5 path:    /home/simon_g/isilon_images_mnt/10_MetaSystems/MetaSystemsData/_simon/data/MCI_data/h5_files/IMC_NB_FineCT/IMC_NB_FineCT.h5
  n_markers:  31   (same as IMC_NeuroblastomaMetaCluster)
  n_classes:  11   (excl. Other)
  patch_size: 24   (same)
  cutter_size:12   (same)
  ignore_annotation: ["Other"]

Cell type counts (excl. Other):
  Tumor            150,531
  T_Cell            49,752
  Stromal           38,454
  Proliferating     31,840
  Myeloid            7,736
  B_Cell             6,715
  Neutrophil         3,917
  CD8_T              3,437
  Progenitor         1,833
  CD4_T              1,830
  NK_DC                111
